In [ ]:
%load_ext autoreload
%autoreload 2

# Vllm flow quickly

In [ ]:
from slurm_ops.core import (
    start_or_connect,
    job_stat,
    update_ssh_node_config,
    find_free_port,
    get_port_forwarding_command,
)

In [ ]:
# ! cp ../ssh_config_templates/* ~/.ssh/
# ! for f in config klone-node-config tillicum-node-config; do sed -i '' 's/deanlcs/<YOUR_CS_ID>/g' ~/.ssh/"$f"; done


In [ ]:
! ssh tillicum-login echo "Connected successfully"

Connected successfully


In [ ]:
job_name = "remote_dev"
slurm_host = "tillicum-login"
start_or_connect(job_name, slurm_host,
    slurm_args="--qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00")

No running job 'remote_dev' found. Starting salloc...
Run this in your terminal:
ssh -t tillicum-login "tmux new-session -A -s remote_dev 'salloc --job-name=remote_dev --qos=debug --gpus=1 --cpus-per-task=8 --mem=200G --time=01:00:00'"


In [ ]:
node, job_id = job_stat(job_name, slurm_host)

Job 'remote_dev' status on tillicum-login:
  Job ID:    63009
  Node:      g014
  Time left: 59:53
  Partition: gpu-h200

Useful commands (run these on the login node or via ssh):
  # Get a shell on the compute node:
  ssh -t tillicum-login 'srun --jobid=63009 --overlap --pty bash'
  # Cancel the job:
  ssh tillicum-login 'scancel 63009'
  # View job details:
  ssh tillicum-login 'scontrol show job 63009'


In [ ]:
update_ssh_node_config(job_name,host=slurm_host)

Updated /Users/deanlight/.ssh/tillicum-node-config: Hostname → g014


'g014'

In [ ]:
! cat /Users/deanlight/.ssh/tillicum-node-config

Host tillicum-node
  User deanlcs
  Hostname g014               
  ProxyJump tillicum-login


In [ ]:
print("uv run bash run_vllm_script.sh")

uv run bash run_vllm_script.sh


In [ ]:
local_port = find_free_port(above=8000)
_=get_port_forwarding_command(local_port=8555, remote_port=vllm_port, node=node, host=slurm_host)

ssh -N -f -L 8555:g007.hyak.local:8555 tillicum-login


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../.envrc') # make sure you have a similar .envrc on your local machine

api_key = os.environ['API_KEY']
model = os.environ['MODEL']

openai_api_base = f"http://localhost:{local_port}/v1"
client = OpenAI(
    api_key=api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(
    model=model,
    prompt="who are you?",
)
print("Completion result:", completion)